In [ ]:
% load_ext autoreload
% autoreload 2

In [ ]:
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.insert(0, r"c:\users\arthu\documents\swittz-epfl\phd\code\lib\site-packages")
sys.path.append('../')
import pymc3 as pm
import theano.tensor as tt
from theano.tensor.nlinalg import matrix_inverse

# Coding Random priors from Orbanz (2014)

## A first go.

In [ ]:
def symmetrized_RBF_kernel(xi1, xi2, s, l, sigma, jitter=1e-4):
    """ Symmetric RBF kernel function.
    Parameters
    ----------
    xi1 : ndarray
        First input.
    xi2 : ndarray
        Second input.
    s : float
        Signal variance.
    l : float
        Length scale.
    sigma : float
        Noise standard deviation.
    add_noise : bool
        Flag to indicate whether to add noise term.
    jitter : float
        Jitter term for numerical stability, when computing matrix_inverse.
    Returns
    -------
    kernel : ndarray
        Symmetric RBF kernel.
    """
    xi1_expanded = xi1.dimshuffle(0, 'x', 1)  # Shape: (M, 1, 2)
    xi2_expanded = xi2.dimshuffle('x', 0, 1)  # Shape: (1, N, 2)

    xi1_flip = xi1_expanded[:, :, ::-1]  # Flip xi1
    
    # Compute squared Euclidean distance
    sqdist1 = tt.sum(tt.square(xi1_expanded - xi2_expanded), axis=2)
    sqdist2 = tt.sum(tt.square(xi1_flip - xi2_expanded), axis=2)

    # Apply RBF kernel
    k1 = s**2 * tt.exp(-sqdist1 / (2 * l**2))
    k2 = s**2 * tt.exp(-sqdist2 / (2 * l**2))

    # Symmetrized kernel
    kernel = 0.5 * (k1 + k2)

    # Add noise term only if xi1 and xi2 are the same (using Theano's allclose function)
    if tt.allclose(xi1, xi2):
        print("Adding noise term")
        kernel += sigma**2 * tt.eye(xi1.shape[0]) + jitter * tt.eye(xi1.shape[0])

    return kernel

In [ ]:
def graphon_model(adj_matrix: np.ndarray,
                  num_pseudoinputs: int = 100):
    """ Construct the graphon model.

    Args:
        adj_matrix: (N, N) adjacency matrix
        num_pseudoinputs: number of pseudoinputs for SoR approximation, corresponds to k in the paper
    Returns:
        model: PyMC3 model
        trace: PyMC3 trace
    """
    N = adj_matrix.shape[0]

    with pm.Model() as model:
        # Latent variables U
        U = pm.Normal('U', mu=0, sigma=1, shape=(N, 1))

        # Kernel parameters
        s = pm.HalfNormal('s', sigma=1)
        l = pm.HalfNormal('l', sigma=1)
        sigma_noise = pm.HalfNormal('sigma_noise', sigma=1)

        # Pseudoinputs for SoR approximation
        eta = pm.Normal('eta', mu=0, sigma=1, shape=(num_pseudoinputs, 2))
        # Constructing the kernel matrix
        K_eta_eta = symmetrized_RBF_kernel(eta, eta, s, l, sigma_noise)
        K_xi_eta = symmetrized_RBF_kernel(U, eta, s, l, sigma_noise)

        T = pm.MvNormal('T', mu=0, cov=K_eta_eta, shape=num_pseudoinputs)



        # Graphon approximation
        W_approx = pm.Deterministic('W_approx', tt.dot(tt.dot(K_xi_eta, matrix_inverse(K_eta_eta)), T))

        # Model the observed data
        X_obs = pm.Normal('X_obs', mu=W_approx, sigma=sigma_noise, observed=adj_matrix)

        # Inference
        trace = pm.sample(1000, return_inferencedata=True)
    
    return model, trace

In [ ]:
def graphon_model2(adj_matrix: np.ndarray,
                  num_pseudoinputs: int = 100):
    """ Construct the graphon model.

    Args:
        adj_matrix: (N, N) adjacency matrix
        num_pseudoinputs: number of pseudoinputs for SoR approximation, corresponds to k in the paper
    Returns:
        model: PyMC3 model
        trace: PyMC3 trace
    """
    N = adj_matrix.shape[0]
    O = N * (N - 1) // 2  # Number of observed relations

    with pm.Model() as model:
        U = pm.Normal('U', mu=0, sigma=1, shape=(N, 1))  # Adjusted for 4 dimensions

        # Kernel parameters with log-normal priors
        l = pm.Lognormal('l', mu=1, sigma=0.5)  # Length scale
        s = pm.Lognormal('s', mu=2, sigma=0.5)  # Scale factor
        sigma_noise = pm.Lognormal('sigma_noise', mu=0.1, sigma=0.5)  # Target noise

        # Pseudoinputs for SoR approximation
        eta = pm.Normal('eta', mu=0, sigma=1, shape=(num_pseudoinputs, 2))  # Eta remains 2D

        # Constructing the kernel matrix
        K_eta_eta = symmetrized_RBF_kernel(eta, eta, s, l, sigma_noise)
        K_xi_eta = symmetrized_RBF_kernel(U, eta, s, l, sigma_noise)

        T = pm.MvNormal('T', mu=0, cov=K_eta_eta, shape=num_pseudoinputs)



        # Graphon approximation
        W_approx = pm.Deterministic('W_approx', tt.dot(tt.dot(K_xi_eta, matrix_inverse(K_eta_eta)), T))

        # Model the observed data
        X_obs = pm.Normal('X_obs', mu=W_approx, sigma=sigma_noise, observed=adj_matrix)

        # Inference
        trace = pm.sample(1000, return_inferencedata=True)
    
    return model, trace

In [ ]:
# Example usage
adj_matrix = np.array([[0, 1, 0], [1, 0, 1], [0, 1, 0]])  # Example adjacency matrix
model, trace = graphon_model2(adj_matrix, num_pseudoinputs=100)


In [ ]:
print(type(trace))

In [ ]:
def compute_connectivity_matrix(trace: pm.backends.base.InferenceData,
                                N: int,
                                num_pseudoinputs: int = 100):
    """ Compute the connectivity matrix from the trace.

    Args:
        trace: PyMC3 trace
        N: number of nodes
        num_pseudoinputs: number of pseudoinputs for SoR approximation
    Returns:
        connectivity_matrix: (N, N) connectivity matrix
    """
    # Compute the mean of the posterior distributions
    s_mean = np.mean(trace.posterior['s'].values)
    l_mean = np.mean(trace.posterior['l'].values)
    sigma_noise_mean = np.mean(trace.posterior['sigma_noise'].values)
    U_mean = np.mean(trace.posterior['U'].values, axis=(0, 1))
    eta_mean = np.mean(trace.posterior['eta'].values, axis=(0, 1))
    T_mean = np.mean(trace.posterior['T'].values, axis=(0, 1))

    # Compute the kernel matrices
    K_eta_eta_mean = symmetrized_RBF_kernel(eta_mean, eta_mean, s_mean, l_mean, sigma_noise_mean).eval()
    K_xi_eta_mean = symmetrized_RBF_kernel(U_mean, eta_mean, s_mean, l_mean, sigma_noise_mean).eval()

    # Compute the connectivity matrix
    connectivity_matrix = np.dot(np.dot(K_xi_eta_mean, np.linalg.inv(K_eta_eta_mean)), T_mean)

    return connectivity_matrix

In [ ]:
connectivity_matrix = compute_connectivity_matrix(trace, N=adj_matrix.shape[0], num_pseudoinputs=5)

## A second go.

In [ ]:
import pymc3 as pm
import numpy as np
import theano.tensor as tt

# Define the RBF kernel function
def rbf_kernel(xi1, xi2, s, l):
    return s**2 * tt.exp(-tt.sum((xi1 - xi2)**2) / (2 * l**2))

# Define the symmetric kernel function with noise
def sym_kernel(xi1, xi2, psi):
    s, l, sigma = psi
    return 0.5 * (rbf_kernel(xi1, xi2, s, l) + rbf_kernel(xi1, tt.set_subtensor(xi2[:, ::-1], 1 - xi2[:, ::-1]), s, l)) + sigma**2 * tt.eye(xi1.shape[0])

In [ ]:
# Define the model
def graphon_model(adj_matrix, n_pseudo_inputs=50):
    
    # Number of nodes in the graph
    N = adj_matrix.shape[0]
    with pm.Model() as model:
        # Uniform latent variables
        U = pm.Uniform('U', shape=(N, 2), testval=np.random.rand(N, 2))

        # Kernel parameters
        sigma = pm.HalfNormal('sigma', sigma=1.)
        length_scale = pm.HalfNormal('length_scale', sigma=1.)
        scale_factor = pm.HalfNormal('scale_factor', sigma=1.)

        # Kernel matrices
        K_xx = sym_kernel(U, U, (scale_factor, length_scale, sigma))
        K_xe = sym_kernel(U, pm.math.stack([U[:, 0], U[:, 1]], axis=-1), (scale_factor, length_scale, sigma))
        K_ee = sym_kernel(pm.math.stack([U[:, 0], U[:, 1]], axis=-1), pm.math.stack([U[:, 0], U[:, 1]], axis=-1), (scale_factor, length_scale, sigma))

        # Pseudo inputs
        pseudo_inputs = pm.Normal('pseudo_inputs', mu=0., sigma=1., shape=(n_pseudo_inputs, 2))

        # Target values
        target_values = pm.Normal('target_values', mu=0., sigma=1., shape=n_pseudo_inputs, observed=np.random.randn(n_pseudo_inputs))

        # Graphon
        K_xe_reshaped = tt.reshape(K_xe, (N, n_pseudo_inputs, 1))
        W = pm.Deterministic('W', tt.sum(tt.dot(K_xe_reshaped, target_values), axis=1))

        # Observed adjacency matrix
        adjacency = pm.Bernoulli('adjacency', p=W, observed=adj_matrix)

    return pm.Model(locals())

In [ ]:
# Example usage
adj_matrix = np.array([[0, 1, 1, 0],
                       [1, 0, 0, 1],
                       [1, 0, 0, 1],
                       [0, 1, 1, 0]], dtype=int)

model = graphon_model(adj_matrix)
trace = pm.sample(1000, tune=1000)

# Build the connectivity matrix
U_i = trace['U'][:, 0]
U_j = trace['U'][:, 1]
N = adj_matrix.shape[0]
connectivity_matrix = trace['W'].reshape(N, N)

## A third go

In [ ]:
import pymc3 as pm
import numpy as np
import theano.tensor as tt
import theano

def rbf_kernel(xi, xj, s, ell):
    """
    Radial Basis Function (RBF) kernel.
    """
    sqdist = tt.sum(tt.square(xi[:, None, :] - xj[None, :, :]), axis=2)
    return s**2 * tt.exp(-0.5 * sqdist / ell**2)

def symmetric_kernel(xi, xj, s, ell, sigma):
    """
    Symmetric kernel with noise.
    """
    xi_swapped = xi[:, [1, 0]]  # Swap coordinates for symmetry
    kappa_bar_xi_xj = rbf_kernel(xi, xj, s, ell)
    kappa_bar_xi_swapped_xj = rbf_kernel(xi_swapped, xj, s, ell)
    return 0.5 * (kappa_bar_xi_xj + kappa_bar_xi_swapped_xj) + tt.eye(xi.shape[0]) * sigma**2

In [ ]:
def compute_K_xi_eta(U, eta, s, ell):
    """
    Computes the cross-covariance matrix K_{xi eta} between the latent positions U and pseudo-inputs eta.
    """
    n = U.shape[0]
    xi = tt.concatenate([U.repeat(n, axis=0), tt.tile(U, (n, 1))], axis=1)
    return symmetric_kernel(xi, eta, s, ell)

In [ ]:
def model_graphon(adjacency_matrix, k):
    n = adjacency_matrix.shape[0]

    with pm.Model() as model:
        # Latent uniform variables for each node
        U = pm.Uniform('U', lower=0, upper=1, shape=(n, 2))

        # GP hyperparameters, values from Orbanz (2014)
        ell = pm.LogNormal('ell', mu=1, sd=0.5)
        s = pm.LogNormal('s', mu=2, sd=0.5)
        sigma = pm.LogNormal('sigma', mu=np.log(0.1), sd=0.5)

        # Pseudo-inputs (eta)
        eta = pm.Normal('eta', mu=0, sd=2, shape=(k, 2))

        # Compute the kernel matrix for eta
        K_eta_eta = symmetric_kernel(eta, eta, s, ell, sigma)

        # Approximate W using the SoR method
        K_xi_eta = compute_K_xi_eta(U, eta, s, ell)
        K_eta_eta_inv = tt.nlinalg.matrix_inverse(K_eta_eta)
        T = pm.MvNormal('T', mu=np.zeros(k), cov=K_eta_eta, shape=(k,))
        W_approx = pm.Deterministic('W_approx', tt.dot(tt.dot(K_xi_eta, K_eta_eta_inv), T))

        # Use W_approx directly for observed adjacency matrix
        # Since W_approx represents probabilities for each edge, it's used directly
        # to define the likelihood of observing edges. We only consider upper triangle,
        # to conform with undirected, no self-loop graph
        observed_edges = tt.triu(adjacency_matrix, k=1).nonzero_values()  # Extract observed edges
        likelihood = pm.Bernoulli('obs', p=W_approx, observed=observed_edges)
    
    return model

In [ ]:
# Example usage
n = 10  # Number of nodes
k = 5   # Number of pseudo-inputs
adjacency_matrix = np.random.randint(0, 2, size=(n, n))
np.fill_diagonal(adjacency_matrix, 0)  # Ensure no self-loops

model = model_graphon(adjacency_matrix, k)

In [ ]:
# Fit the model with n_samples MCMC samples
n_samples = 1000
with model:
    trace = pm.sample(n_samples, return_inferencedata=False)

In [ ]:
W_approx_samples = trace['W_approx']
# average over samples
mean_W_approx = np.mean(W_approx_samples, axis=0)
# reshape to adjacency matrix
pij = np.zeros((n, n))
pij[np.triu_indices(n, k=1)] = mean_W_approx
pij += pij.T

# Hoff (2007)

In [ ]:
import pymc3 as pm
import theano.tensor as tt
import numpy as np

In [ ]:
import theano.tensor as tt

def alpha_latent_class(u_i, u_j, M):
    return M[u_i, u_j]

def alpha_latent_distance(u):
    # Calculate the pairwise squared differences
    diff = u[:, None, :] - u[None, :, :]
    squared_diff = tt.sum(diff**2, axis=-1)

    return -tt.sqrt(squared_diff)


def alpha_latent_eigenmodel(u_i, u_j, Lambda):
    return tt.dot(u_i, Lambda * u_j)

In [ ]:
import os
os.environ["THEANO_FLAGS"] = "optimizer=fast_compile"

def build_model(n, K, X, y, model_type):
    with pm.Model() as model:
        # Priors for parameters
        mu = pm.Normal('mu', mu=0, sigma=1)
        beta = pm.Normal('beta', mu=0, sigma=1, shape=X.shape[2])

        # Initialize Lambda and M
        Lambda = M = None

        # Define latent variables and alpha function based on model type
        if model_type == 'latent_class':
            M = pm.Normal('M', mu=0, sigma=1, shape=(K, K))
            u = pm.Categorical('u', p=np.ones(K)/K, shape=n)
            alpha = alpha_latent_class
        elif model_type == 'latent_distance':
            u = pm.Normal('u', mu=0, sigma=1, shape=(n, K))
            alpha = alpha_latent_distance
        elif model_type == 'latent_eigenmodel':
            Lambda = pm.Normal('Lambda', mu=0, sigma=1, shape=(K, K))
            u = pm.Normal('u', mu=0, sigma=1, shape=(n, K))
            alpha = alpha_latent_eigenmodel

        # Model likelihood
        theta = tt.zeros((n, n))
        # Inside build_model function
        if model_type == 'latent_distance':
            alpha_ij_matrix = alpha(u)
            for i in range(n):
                for j in range(i+1, n):
                    theta_ij = pm.math.invlogit(mu + tt.dot(X[i, j], beta) + alpha_ij_matrix[i, j])
                    theta = tt.set_subtensor(theta[i, j], theta_ij)
                    theta = tt.set_subtensor(theta[j, i], theta_ij)  # Symmetry
        else:
            for i in range(n):
                for j in range(i+1, n):
                    if model_type == 'latent_class':
                        alpha_ij = alpha(u[i], u[j], M)
                    elif model_type == 'latent_eigenmodel':
                        alpha_ij = alpha(u[i], u[j], Lambda)
                    theta_ij = pm.math.invlogit(mu + tt.dot(X[i, j], beta) + alpha_ij)
                    theta = tt.set_subtensor(theta[i, j], theta_ij)
                    theta = tt.set_subtensor(theta[j, i], theta_ij)  # Symmetry


        Y_obs = pm.Bernoulli('Y_obs', p=theta, observed=y)

    return model, theta


In [ ]:
import numpy as np

n = 10  # Number of nodes
K = 2   # Number of classes or dimensions
X = np.random.randn(n, n, 2)  # Predictor variables

# Generate a symmetric binary network data matrix
upper_triangle_indices = np.triu_indices(n, k=1)
y = np.zeros((n, n), dtype=int)
y[upper_triangle_indices] = np.random.randint(0, 2, size=len(upper_triangle_indices[0]))
y += y.T  # Make the matrix symmetric

# Ensure the diagonal remains zero (self-connections are not considered in simple graphs)
np.fill_diagonal(y, 0)

print("min and max of y: ", np.min(y), np.max(y))
print("min and max of X: ", np.min(X), np.max(X))

model_type = 'latent_distance'  # Choose 'latent_class', 'latent_distance', or 'latent_eigenmodel'
model, pij = build_model(n, K, X, y, model_type)

with model:
    trace = pm.sample(1000)
    pm.summary(trace)
